# Formatos, importação e extração

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_03/01_formatos_importacao_e_extracao.ipynb)

## 1. Formato é estrutura e affordance

CSV registra uma tabela sem fórmulas ou tipos ricos; XLSX pode conter várias
planilhas, fórmulas e formatação; JSON e XML expressam hierarquias; TXT não
define internamente como interpretar seu conteúdo; PDF busca preservar uma
apresentação de página e pode conter texto, imagem ou ambos. Extensão não
garante conteúdo nem qualidade.

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_03'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = [('openpyxl', 'openpyxl>=3.1,<4'), ('pypdf', 'pypdf>=5,<6'), ('PIL', 'Pillow>=11,<12')]
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

In [ ]:
import json
from pathlib import Path
from xml.etree import ElementTree as ET
import pandas as pd

pasta = Path("dados/brutos")
csv_df = pd.read_csv(pasta / "catalogo_messy.csv", sep=";")
xlsx_df = pd.read_excel(pasta / "catalogo_messy.xlsx", sheet_name="documentos")
json_data = json.loads((pasta / "metadados.json").read_text(encoding="utf-8"))
xml_raiz = ET.parse(pasta / "metadados.xml").getroot()
txt = (pasta / "D001.txt").read_text(encoding="utf-8")

print("CSV/XLSX iguais nas dimensões:", csv_df.shape == xlsx_df.shape)
print("Registros JSON:", len(json_data), "| XML:", len(xml_raiz))
print("TXT:", txt.strip())

### Interpretação

Leitores diferentes retornam objetos diferentes. Importar com sucesso não
prova que encoding, separador, planilha, hierarquia ou tipos foram interpretados
corretamente. Registre parâmetros, versão da fonte e testes esperados.

## 2. Base pública sem dependência de rede

O arquivo `extrato_codigos_municipios_ibge.csv` é uma cópia local pequena da
tabela do IBGE. O JSON de proveniência registra página, acesso e recorte. Uma
base pública muda; por isso, citar “IBGE” sem versão/data não basta.

In [ ]:
municipios = pd.read_csv(
    pasta / "extrato_codigos_municipios_ibge.csv",
    dtype={"codigo_municipio": "string"},
)
proveniencia = json.loads(
    (pasta / "proveniencia_base_publica.json").read_text(encoding="utf-8")
)
print(proveniencia["fonte"], "— acesso:", proveniencia["data_de_acesso"])
municipios

## 3. Extração de PDF não é OCR

Se o PDF contém caracteres, um leitor pode extrair a camada textual. Se cada
página é apenas imagem, a extração retorna pouco ou nada e será necessário OCR.
Mesmo PDF com texto pode ter ordem de leitura problemática, hifenização ou
caracteres incorretos.

![Um arquivo PDF passa pelo teste de texto selecionável: quando há texto, extrai-se a camada textual; quando não há, aplica-se OCR; as duas rotas exigem avaliação contra a página.](imagens/01_pdf_texto_imagem_ocr.svg)

O diagrama evita uma confusão frequente: OCR não é sinônimo de “abrir PDF”.
Primeiro se diagnostica o conteúdo; depois se escolhe e avalia a operação.

In [ ]:
from pypdf import PdfReader

leitor = PdfReader(pasta / "documento_textual.pdf")
texto_pdf = "\n".join(pagina.extract_text() or "" for pagina in leitor.pages)
print(texto_pdf.strip())

## 4. OCR como hipótese de transcrição

OCR reconhece caracteres em imagem. Resolução, inclinação, ruído, layout,
fonte e idioma afetam o resultado. A documentação do Tesseract recomenda
inspecionar e preparar imagens quando necessário. A transcrição deve ser
vinculada à imagem, à ferramenta, aos parâmetros e a uma avaliação de erro.

As duas imagens abaixo são **simulações didáticas**, não documentos
históricos. Elas contêm a mesma linha: a primeira está limpa; a segunda foi
reduzida, ampliada, inclinada, desfocada e marcada de forma controlada.

![Linha tipográfica sintética limpa usada como referência no experimento controlado de reconhecimento óptico de caracteres.](dados/brutos/pagina_digitalizada.png)

![A mesma linha tipográfica sintética após redução, ampliação, inclinação, desfoque e marcas controladas que dificultam o reconhecimento óptico de caracteres.](dados/brutos/pagina_digitalizada_degradada.png)

In [ ]:
import shutil
import subprocess

entradas_ocr = [
    ("imagem limpa", pasta / "pagina_digitalizada.png", pasta / "ocr_precomputado.txt"),
    ("imagem degradada", pasta / "pagina_digitalizada_degradada.png", pasta / "ocr_precomputado_degradado.txt"),
]

def reconhecer_ou_carregar(imagem, precomputado):
    if shutil.which("tesseract"):
        try:
            processo = subprocess.run(
                ["tesseract", str(imagem), "stdout", "-l", "eng", "--psm", "7"],
                capture_output=True, text=True, check=True,
            )
            return processo.stdout.strip(), "Tesseract executado agora"
        except subprocess.CalledProcessError:
            pass
    return (
        precomputado.read_text(encoding="utf-8").strip(),
        "saída pré-computada fornecida com o material",
    )

resultados_ocr = []
for condicao, imagem, precomputado in entradas_ocr:
    transcricao, origem = reconhecer_ou_carregar(imagem, precomputado)
    resultados_ocr.append({"condição": condicao, "transcrição": transcricao, "origem": origem})

pd.DataFrame(resultados_ocr)

In [ ]:
referencia = (pasta / "ocr_referencia.txt").read_text(encoding="utf-8").strip()

def distancia_edicao(a, b):
    anterior = list(range(len(b) + 1))
    for i, item_a in enumerate(a, 1):
        atual = [i]
        for j, item_b in enumerate(b, 1):
            atual.append(min(
                atual[-1] + 1,
                anterior[j] + 1,
                anterior[j - 1] + (item_a != item_b),
            ))
        anterior = atual
    return anterior[-1]

comparacao = []
for resultado in resultados_ocr:
    observado = resultado["transcrição"]
    comparacao.append({
        "condição": resultado["condição"],
        "erros em caracteres": distancia_edicao(referencia, observado),
        "CER": distancia_edicao(referencia, observado) / len(referencia),
        "erros em palavras": distancia_edicao(referencia.split(), observado.split()),
        "WER": distancia_edicao(referencia.split(), observado.split()) / len(referencia.split()),
    })

pd.DataFrame(comparacao).round({"CER": 3, "WER": 3})

### Como interpretar a comparação

CER é a taxa de erro por caractere; WER, por palavra. Zero significa
coincidência com a transcrição de referência nesta amostra. Valores maiores
indicam mais edições necessárias, mas **não decidem sozinhos** se o texto é
adequado: busca exploratória, citação e análise lexical exigem tolerâncias
diferentes. Uma referência também pode conter erro humano; por isso, a
página e o protocolo de transcrição continuam indispensáveis.

## Atividade — inventário técnico

Para cada fonte do projeto, registre formato, estrutura interna, leitor,
encoding/planilha/nó, presença de texto, necessidade de OCR, riscos, teste e
saída prevista. Diferencie claramente dado recebido, texto extraído e texto
reconhecido. **Inventário:** Escreva aqui.

## Síntese

A importação é interpretação técnica. O objetivo não é converter tudo para um
único formato sem crítica, mas produzir representações adequadas, vinculadas e
testáveis.